# Eigenshirts: PCA + OLS vs log-loss classifiers (jerseys vs shirts)

Run all cells (Runtime → Run all). Screenshot the outputs of the OLS and Log Loss cells for the upload question.

In [ ]:
# Download and unzip the data (runs in Colab)
!wget -q -nc https://course-resources.minerva.kgi.edu/uploaded_files/mke/YpANOV/jerseys.zip
!wget -q -nc https://course-resources.minerva.kgi.edu/uploaded_files/mke/rQb8dn/shirts.zip
!unzip -q -o jerseys.zip -d jerseys
!unzip -q -o shirts.zip -d shirts
!find jerseys shirts -type f | head
!echo "jerseys: $(find jerseys -type f | wc -l) files, shirts: $(find shirts -type f | wc -l) files"


In [ ]:
#PCA: load, crop, resize, find k, eigenshirts, reconstructions
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image, ImageOps
from sklearn.decomposition import PCA

SIZE = 64                                  # every image becomes 64 x 64 x 3
FOLDERS = {"jerseys": 0, "shirts": 1}      # folder name -> label
MAX_PER_CLASS = 150
EXTS = {".jpg", ".jpeg", ".png"}

def load_image(path):
    img = Image.open(path).convert("RGB")
    img = ImageOps.fit(img, (SIZE, SIZE))  # center crop to square, then resize
    return np.asarray(img, dtype=np.float32) / 255.0

images, labels = [], []
for folder, label in FOLDERS.items():
    paths = sorted(p for p in Path(folder).rglob("*")
                   if p.suffix.lower() in EXTS and "__MACOSX" not in p.parts
                   and not p.name.startswith("."))[:MAX_PER_CLASS]
    for p in paths:
        try:
            images.append(load_image(p))
            labels.append(label)
        except Exception:
            pass                           # skip broken files

images = np.array(images)
y = np.array(labels)
X = images.reshape(len(images), -1)        # flatten: one row per image
print("images:", images.shape, " X:", X.shape, " per class:", np.bincount(y))

pca_full = PCA().fit(X)
cum_var = np.cumsum(pca_full.explained_variance_ratio_)
k = int(np.argmax(cum_var >= 0.90) + 1)    # smallest k that keeps 90% of variance
print("components for 90% variance:", k)

plt.plot(cum_var)
plt.axhline(0.90, color="r", linestyle="--")
plt.axvline(k, color="r", linestyle="--")
plt.xlabel("number of components"); plt.ylabel("cumulative explained variance")
plt.show()

def to_img(v):
    v = v.reshape(SIZE, SIZE, 3)
    return (v - v.min()) / (v.max() - v.min())   # rescale to 0-1 so it displays

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes[0, 0].imshow(pca_full.mean_.reshape(SIZE, SIZE, 3)); axes[0, 0].set_title("mean")
for i, ax in enumerate(axes.flat[1:]):
    ax.imshow(to_img(pca_full.components_[i])); ax.set_title(f"PC {i + 1}")
for ax in axes.flat:
    ax.axis("off")
plt.show()

def reconstruct(n):
    p = PCA(n).fit(X)
    return np.clip(p.inverse_transform(p.transform(X)), 0, 1)

rec_10, rec_k = reconstruct(10), reconstruct(k)
idx = np.random.RandomState(0).choice(len(X), 6, replace=False)
fig, axes = plt.subplots(3, 6, figsize=(12, 6))
for col, i in enumerate(idx):
    axes[0, col].imshow(images[i])
    axes[1, col].imshow(rec_10[i].reshape(SIZE, SIZE, 3))
    axes[2, col].imshow(rec_k[i].reshape(SIZE, SIZE, 3))
axes[0, 0].set_ylabel("original"); axes[1, 0].set_ylabel("10 PCs"); axes[2, 0].set_ylabel(f"{k} PCs")
for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([])
plt.show()

Z = pca_full.transform(X)
for label, name in [(0, "jerseys"), (1, "shirts")]:
    plt.scatter(Z[y == label, 0], Z[y == label, 1], alpha=0.6, label=name)
plt.xlabel("PC 1"); plt.ylabel("PC 2"); plt.legend()
plt.show()


In [ ]:
# OLS
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, confusion_matrix

# Same split for both cells, stratified so both classes are balanced in train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y)

def ols_classifier(n_components):
    """PCA fitted on training data only, then OLS on the PCA scores, threshold at 0.5."""
    if n_components is None:                      # raw pixels, no PCA
        Z_train, Z_test = X_train, X_test
    else:
        pca = PCA(n_components).fit(X_train)
        Z_train, Z_test = pca.transform(X_train), pca.transform(X_test)
    ols = LinearRegression().fit(Z_train, y_train)   # minimizes sum of squared errors
    pred_train = (ols.predict(Z_train) >= 0.5).astype(int)
    pred_test = (ols.predict(Z_test) >= 0.5).astype(int)
    return accuracy_score(y_train, pred_train), accuracy_score(y_test, pred_test), pred_test

print("OLS (squared loss) classifier")
print(f"{'components':>12} {'train acc':>10} {'test acc':>10}")
for n in [2, 5, 10, 20, 50, k, None]:
    tr, te, _ = ols_classifier(n)
    name = "raw pixels" if n is None else str(n)
    print(f"{name:>12} {tr:>10.3f} {te:>10.3f}")

_, _, pred_k = ols_classifier(k)
print(f"\nConfusion matrix with k = {k} components (rows = true, cols = predicted):")
print(confusion_matrix(y_test, pred_k))


In [ ]:
# Log Loss
# Uses X_train, X_test, y_train, y_test from the OLS cell so both models see the same split.
import numpy as np
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, confusion_matrix

def logistic_classifier(n_components):
    """PCA fitted on training data only, then logistic regression (minimizes log loss)."""
    if n_components is None:
        Z_train, Z_test = X_train, X_test
    else:
        pca = PCA(n_components).fit(X_train)
        Z_train, Z_test = pca.transform(X_train), pca.transform(X_test)
    clf = LogisticRegression(max_iter=5000).fit(Z_train, y_train)
    p_test = clf.predict_proba(Z_test)[:, 1]
    return (accuracy_score(y_train, clf.predict(Z_train)),
            accuracy_score(y_test, clf.predict(Z_test)),
            log_loss(y_test, p_test),
            clf.predict(Z_test))

print("Logistic regression (log loss) classifier")
print(f"{'components':>12} {'train acc':>10} {'test acc':>10} {'test logloss':>13}")
for n in [2, 5, 10, 20, 50, k, None]:
    tr, te, ll, _ = logistic_classifier(n)
    name = "raw pixels" if n is None else str(n)
    print(f"{name:>12} {tr:>10.3f} {te:>10.3f} {ll:>13.3f}")

_, _, _, pred_k = logistic_classifier(k)
print(f"\nConfusion matrix with k = {k} components (rows = true, cols = predicted):")
print(confusion_matrix(y_test, pred_k))
